# A simple notebook to perform binary classification on a comprehensive pokemon dataset to figure whether a given specimen is legendary or not
# Dataset taken from Kaggle - https://www.kaggle.com/datasets/rounakbanik/pokemon

In [ ]:
# ----------------------------------------------------------------------------------------
# Import basic libraries
# ----------------------------------------------------------------------------------------
import numpy as np
import pandas as pd
import ast

# ----------------------------------------------------------------------------------------
# Import graphic libraries
# ----------------------------------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# ----------------------------------------------------------------------------------------
# Import scikit-learn libraries
# ----------------------------------------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Project Constants

In [ ]:
# ----------------------------------------------------------------------------------------
# global constants
# ----------------------------------------------------------------------------------------
random_seed = 42

# ----------------------------------------------------------------------------------------
# dataset constants
# ----------------------------------------------------------------------------------------
dataset_cvs_filename = "pokemon.csv"
dataset_cvs_delimiter_char = ','
dataset_cvs_num_rows_read = None # specify 'None' if want to read whole file

label_classification = 'is_legendary'

# ----------------------------------------------------------------------------------------
# machine learning constants
# ----------------------------------------------------------------------------------------
common_cv_folds = 5
common_scoring_metric = 'accuracy'

# train test split constants
test_train_split_rate = 0.2
test_train_split_random_state = random_seed

# random forrest constants
rf_random_state = random_seed
rf_min_samples_split = 2
rf_verbose = 1

# grid search random forrest constants 
rf_grid_search_cv_folds = common_cv_folds
rf_grid_search_scoring = common_scoring_metric
rf_grid_search_random_state = random_seed
rf_grid_search_verbose = 1

# XGBoost constants
xgboost_random_state = random_seed

# Grid search XGBoost constants
xgboost_grid_search_cv_folds = common_cv_folds
xgboost_grid_search_scoring = common_scoring_metric
xgboost_grid_search_random_state = random_seed

## Project Utility Routines

In [ ]:
# Display Dataset Distribution Graphs (histogram/bar graph) of column data
def plotPerColumnDistribution(df, nGraphShown, nGraphPerRow):
    nunique = df.nunique()
    df = df[[col for col in df if nunique[col] > 1 and nunique[col] < 50]] # For displaying purposes, pick columns that have between 1 and 50 unique values
    nRow, nCol = df.shape
    columnNames = list(df)
    nGraphRow = int((nCol + nGraphPerRow - 1) / nGraphPerRow)
    plt.figure(num = None, figsize = (6 * nGraphPerRow, 8 * nGraphRow), dpi = 80, facecolor = 'w', edgecolor = 'k')
    for i in range(min(nCol, nGraphShown)):
        plt.subplot(nGraphRow, nGraphPerRow, i + 1)
        columnDf = df.iloc[:, i]
        if (not np.issubdtype(type(columnDf.iloc[0]), np.number)):
            valueCounts = columnDf.value_counts()
            valueCounts.plot.bar()
        else:
            columnDf.hist()
        plt.ylabel('counts')
        plt.xticks(rotation = 90)
        plt.title(f'{columnNames[i]} (column {i})')
    plt.tight_layout(pad = 1.0, w_pad = 1.0, h_pad = 1.0)
    plt.show()

In [ ]:
# Dataset Correlation matrix
def plotCorrelationMatrix(df, graphWidth, title):
    df = df.dropna(axis='columns') # drop columns with NaN
    df = df[[col for col in df if df[col].nunique() > 1]] # keep columns where there are more than 1 unique values
    if df.shape[1] < 2:
        print(f'No correlation plots shown: The number of non-NaN or constant columns ({df.shape[1]}) is less than 2')
        return
    corr = df.corr()
    plt.figure(num=None, figsize=(graphWidth, graphWidth), dpi=80, facecolor='w', edgecolor='k')
    corrMat = plt.matshow(corr, fignum = 1)
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.gca().xaxis.tick_bottom()
    plt.colorbar(corrMat)
    plt.title(f'Correlation Matrix for {title}', fontsize=15)
    plt.show()

## Dataset Loading and Exploration

In [ ]:
# load dataset from cvs file
try:
    df = pd.read_csv(dataset_cvs_filename, delimiter=dataset_cvs_delimiter_char, nrows = dataset_cvs_num_rows_read)
    num_rows, num_columns = df.shape
except Exception as e:
    print("⚠️ Unable to load Pokemon dataset.")
    exit(0)

print(f"✅ Pokemon dataset has been succefully loaded: there are {num_rows} rows and {num_columns} columns.\n")

In [ ]:
# sanity checks for the classification column
df_labels = df[label_classification]

if df_labels.isnull().sum() != 0:
    print(f"⚠️ The Pokemon dataset is not actionable, there are null value in the '{label_classification}' column.")
    exit(0)
if df_labels.dtype != 'int64':
    print(f"⚠️ The Pokemon dataset is not actionable, there are null value in the '{label_classification}' column.")
    exit(0)

num_pokemon_legendary = df_labels.value_counts().get(1)
num_pokemon_non_legendary = df_labels.value_counts().get(0)

if num_pokemon_legendary + num_pokemon_non_legendary != len(df):
    print(f"⚠️ The Pokemon dataset is not actionable, the '{label_classification}' column contains values different than zero or one.")

print(f"✅ There are {num_pokemon_legendary} legendary pokemon in the dataset ({num_pokemon_legendary/len(df)*100:.2f}%).")

In [ ]:
# display first rows
df.head()

In [ ]:
# features
df_features = df.drop(label_classification, axis=1)

# display numerical and categorical columns 
numerical_features = df_features.select_dtypes(include=['int64', 'float64']).columns
print(f"Numerical Features: {numerical_features}.")

categorical_features = df_features.select_dtypes(include=['object']).columns
print(f"Categorical Features: {categorical_features}.")

## First observations from dataset exploration: 
    - 'japanese_name', 'name' : colmuns will be dropped, japanaese names or names should not influence legendary classification
    - 'pokedex_number' : column will be dropped, index of appearance in pokemodex should not influence legendary classification 
    - 'capture_rate' : need to check values, should be a numerical column
    - 'abilities' : multi-values string describing 'features' of pokemons. These information need therefore special processing to be transformed to proper column 

In [ ]:
# Explore the null values in the dataset
df.isnull().sum().sort_values(ascending=False)

## Second observations from dataset exploration: 
    - 'type2' : the colmun will be dropped, this is a sub-type ('string' type) of pokemon so imputation strategy is irrelevant
    - 'percentage_male' : null values will be replaced with a neutral value, i.e. 50.0
    - 'weight_kg', 'height_m' : null value wil be replaced with average value

In [ ]:
# drop unactionable columns for prediction
actionable_df_features = df_features.drop(['japanese_name', 'name', 'type2', 'pokedex_number'], axis=1)
actionable_num_rows, actionable_num_columns = actionable_df_features.shape

print(f"✅ The Pokemon actionable dataset has {actionable_num_columns} feature colmuns.")

# recompute numerical and categorical features 
actionable_numerical_features = actionable_df_features.select_dtypes(include=['int64', 'float64']).columns
print(f"Actionable numerical Features: {actionable_numerical_features}.")

actionable_categorical_features = actionable_df_features.select_dtypes(include=['object']).columns
print(f"Actionable categorical Features: {actionable_categorical_features}.")

In [ ]:
# check unexpected typed column 'capture_rate' : type is 'object' but should it be 'int64' or 'float64' instead?
df_capture_rate = actionable_df_features['capture_rate'].info()

In [ ]:
# check wether 'capture_rate' values are actually numerical, if not replace them by None
for idx in range(0, len(actionable_df_features['capture_rate'])):
    try:
        val = float(actionable_df_features['capture_rate'].at[idx])
    except Exception as e:
        print(f"⚠️ An anomaly has been found in the 'capture_rate' column at index '{idx}': value '{actionable_df_features['capture_rate'].at[idx]}' has been replaced by 'None'.")
        actionable_df_features['capture_rate'].at[idx] = None

print(f"⚠️ Number of anomalies found : {actionable_df_features['capture_rate'].isnull().sum()}")

In [ ]:
# now convert 'capture_rate' column to float64
actionable_df_features['capture_rate'] =  actionable_df_features['capture_rate'].astype('float64')

In [ ]:
# recompute numerical and categorical features 
numerical_features = actionable_df_features.select_dtypes(include=['int64', 'float64']).columns
print(f"Numerical Features: {numerical_features}.")

categorical_features = actionable_df_features.select_dtypes(include=['object']).columns
print(f"Categorical Features: {categorical_features}.")

In [ ]:
# Explore the null values in the dataset
actionable_df_features.isnull().sum().sort_values(ascending=False)

## Third observation from dataset exploration: 
    - 'capture_rate' : null values will be replaced with a neutral value, i.e. 50.0

In [ ]:
# Display correlation matrix of numerical features
plotCorrelationMatrix(actionable_df_features.drop(columns=categorical_features), 8, "pokemon numerical features")

In [ ]:
# Now explore the 'abilities' column
actionable_df_features['abilities']

In [ ]:
num_rows, num_columns = actionable_df_features.shape
print(f"Before the 'abilities' transformation, Pokemon dataset contains {num_rows} rows and {num_columns} columns.")

# Explode the multi-abilities string values into lists of abilities 
actionable_df_features['abilities'] = actionable_df_features['abilities'].apply(ast.literal_eval)

# Generate a dataframe where each column represents a unique abiltity found in the previous 'abilities' column
# join('|') temporarily transforms a list into a '|' delimiter separated string; 
# get_dummies finds all unique elements separately and builds columns with 0 or 1 values
new_abilities = actionable_df_features['abilities'].str.join('|').str.get_dummies()

# Drop the 'abilities' original column 
actionable_df_features = actionable_df_features.drop(labels=['abilities'], axis=1)

# concatenate the new dataframe 
actionable_df_features = pd.concat([actionable_df_features, new_abilities], axis=1)

num_rows, num_columns = actionable_df_features.shape
print(f"✅ After the 'abilities' transformation, actionable Pokemon dataset contains {num_rows} rows and {num_columns} columns.")

In [ ]:
actionable_df_features.head()

In [ ]:
# recompute numerical and categorical features 
numerical_features = actionable_df_features.select_dtypes(include=['int64', 'float64']).columns
print(f"Numerical Features ({len(numerical_features)}): {numerical_features}.")

categorical_features = actionable_df_features.select_dtypes(include=['object']).columns
print(f"Categorical Features ({len(categorical_features)}): {categorical_features}.")

In [ ]:
# Explore the null values in the dataset
actionable_df_features.isnull().sum().sort_values(ascending=False)

## Summary of Observations
    - 'height_m', 'weight_kg' : replace null value by average of column
    - 'percentage_male', 'capture_rate' : replace null value by neutral 50.0 value

In [ ]:
# show distribution of columns
plotPerColumnDistribution(actionable_df_features, 10, 5)

In [ ]:
# dataframe cleaned and preped for training 
X = actionable_df_features
y = df_labels

num_rows, num_columns = X.shape
print(f"✅ The Pokemon dataset has been cleaned and prepared for training, it contains {num_rows} rows and {num_columns} columns.")

# recompute numerical and categorical features 
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns
print(f"    Numerical Features ({len(numerical_features)}): {numerical_features.to_list()}.")

categorical_features = X.select_dtypes(include=['object']).columns
print(f"    Categorical Features ({len(categorical_features)}): {categorical_features.to_list()}.")

## Train / Test Split Pokemon Dataset

In [ ]:
# split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=test_train_split_rate, 
                                                    random_state=test_train_split_random_state)

print(f"✅ Training dataset contains {X_train.shape[0]} rows and {X_train.shape[1]} columns.")

num_pokemon_legendary_train = y_train.value_counts().get(1)
print(f"        There are {num_pokemon_legendary_train} legendary pokemons ({num_pokemon_legendary_train/y_train.shape[0]*100:.2f}%) in the dataset.")

num_pokemon_legendary_test = y_test.value_counts().get(1)
print(f"✅ Test dataset contains {X_test.shape[0]} rows and {X_test.shape[1]} columns.")
print(f"        There are {num_pokemon_legendary_test} legendary pokemons ({num_pokemon_legendary_train/y_train.shape[0]*100:.2f}%) in the dataset.")

## Dataset Cleanup Pipeline
    - categorical_features : performs one-hot encoder
    - numerical_features : performs either 'mean' or 'constant' imputation and scaling

In [ ]:
# create the preparation pipeline
numerical_features_constant = X[['percentage_male', 'capture_rate']].columns
print(f"Numerical Features for constant imputation({len(numerical_features_constant)}): {numerical_features_constant.to_list()}.")

numerical_constant_xformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=50.0)),
    ('scaler', StandardScaler()),
])

numerical_features_mean = numerical_features.drop(['percentage_male', 'capture_rate'])
print(f"Numerical Features for mean imputation({len(numerical_features_mean)}): {numerical_features_mean.to_list()}.")

numerical_mean_xformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
])

print(f"Categorical Features for one-hot imputation({len(categorical_features)}): {categorical_features.to_list()}.")

# ignore new values for onehot encoder
# might error out otherwise if value to encode is found in test dataset but is not present in train dataset
categorical_xformer = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore')),
])

# create the transformer for dataset preparation
preprocessor = ColumnTransformer(
    transformers=[
        ('numerical_constant', numerical_constant_xformer, numerical_features_constant),
        ('numerical_mean', numerical_mean_xformer, numerical_features_mean),
        ('categorical', categorical_xformer, categorical_features)
    ], 
    remainder='passthrough'
)

print(f"✅ Dataset training preprocessor has been created.")

## Random Forrest Model

### Train and predict

In [ ]:
# Create Random Forrest model
random_forest_model = RandomForestClassifier(random_state=rf_random_state, 
                                             min_samples_split=rf_min_samples_split,
                                             verbose=rf_verbose)

# Create full pipleline with Random Forrest model
random_forest_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', random_forest_model)
])

print(f"✅ Random Forrest Training pipeline has been created.")

# train
random_forest_pipeline.fit(X_train, y_train)

print(f"✅ Random Forrest model has been trained using training dataset.")

# predict
y_pred = random_forest_pipeline.predict(X_test)

print(f"✅ Random Forrest prediction has been computed using test dataset.")

### Evaluation

In [ ]:
# Display accuracy score
accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Random Forrest Model Accuracy: {accuracy}")

# Display classification report
print(f"✅ Random Forrest Model Classification Report:")
cr = classification_report(y_test, y_pred)
print(cr)

# Display confusion matrix
print(f"✅ Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Not Legendary', 'Legendary'], 
            yticklabels=['Not Legendary', 'Legendary'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

### Grid Search - Random Forrest 

In [ ]:
# Random Forrest hyper-parameters to optimize
rf_hparams = {
    'n_estimators': [2, 5, 10, 20, 50],
    'max_depth': [None, 2, 5, 10, 20],
    'min_samples_split': [2, 5, 10, 15, 20]
}

# create grid search
grid_search = GridSearchCV(RandomForestClassifier(random_state=rf_random_state), 
                           rf_hparams, 
                           cv=StratifiedKFold(n_splits=rf_grid_search_cv_folds), 
                           scoring=rf_grid_search_scoring, 
                           verbose=rf_grid_search_verbose,
                           n_jobs=-1) # multi-thread 

# grid search exploration
grid_search.fit(preprocessor.fit_transform(X_train), y_train)

print(f"✅ Random Forrest Grid Search completed: ")
print(f"    Best hyper parameters: {grid_search.best_params_}")

# Select Best Random Forrest Model
best_model = grid_search.best_estimator_

# Define full pipleline with the best Random Forrest model
best_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', best_model)
])

### Cross-validation of best Random Forrest model found by grid sreach

In [ ]:
# cross validation results
cv = StratifiedKFold(n_splits=rf_grid_search_cv_folds, 
                     shuffle=True, 
                     random_state=rf_grid_search_random_state)

cv_scores = cross_val_score(best_pipeline, 
                            X_train, y_train, 
                            cv=cv, 
                            scoring=rf_grid_search_scoring)

print(f"cross validation scores - RF default : {cv_scores}")
print(f"Cross validation scores mean - RF default : {np.mean(cv_scores)}")

### Train and predict of best Random Forrest model found in grid search

In [ ]:
# train best pipeline
best_pipeline.fit(X_train, y_train)

# predict best pipeline
y_pred = best_pipeline.predict(X_test)

### Evaluation of best Random Forrest model

In [ ]:
# Display accuracy score
accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Random Forrest Model Accuracy: {accuracy}")

# Display classification report
print(f"✅ Random Forrest Model Classification Report:")
cr = classification_report(y_test, y_pred)
print(cr)

# Display confusion matrix
print(f"✅ Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Not Legendary', 'Legendary'], 
            yticklabels=['Not Legendary', 'Legendary'])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()